In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import graphtools as gt
import phate
import magic
import scprep
import cmocean
import sklearn
import scipy
import seaborn as sns
import meld
sc.settings.set_figure_params(dpi=500, dpi_save=1000, figsize=(5,5), facecolor='white')
sc.settings.verbosity = 3  

In [ ]:
# set output directory
fig_dir = "/dfs7/swaruplab/shared_lab/Collaborations/Wood/Reinstatement_Project/reinstatement_2022/figures/"
data_dir = "/dfs7/swaruplab/shared_lab/Collaborations/Wood/Reinstatement_Project/reinstatement_2022/data/"

sc.settings.figdir = fig_dir

In [ ]:
# load processed data
adata = sc.read_h5ad('{}harmony_processed.h5ad'.format(data_dir))

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
# Check if the 'barcode' column exists in adata.obs
if 'barcode' in adata.obs.columns:
    # Compare the row names (index) with the 'barcode' column
    is_same = (adata.obs.index == adata.obs['barcode']).all()

    if is_same:
        print("The row names of adata.obs are the same as the 'barcode' column.")
    else:
        print("The row names of adata.obs are not the same as the 'barcode' column.")
else:
    print("The 'barcode' column does not exist in adata.obs.")


In [ ]:
adata.obs['batch'].value_counts()

In [ ]:
adata.obs['Assignment'].value_counts()

In [ ]:
adata.obs.groupby('Group')['Assignment'].value_counts()

In [ ]:
adata.obs.groupby('Sample')['Assignment'].value_counts()

In [ ]:
adata.obs.groupby('Sample')['Group'].value_counts()

In [ ]:
adata.obs.groupby('Group')['batch'].value_counts()

### UMAP 

In [ ]:
# plot UMAP to test that it loaded correctly:
sc.pl.umap(adata, color=['annotation'],frameon=False, legend_loc='on data', legend_fontoutline=1, legend_fontsize=9, add_outline=False, title='')


### Run MELD 

In [ ]:
# subset anndata by Behavior only

adata.obs.Group.value_counts()

# adata = adata[adata.obs.Group.isin(['Nurr2c', 'GFP'])].copy()
# adata.shape

In [ ]:
adata.obs.Sample.value_counts()


In [ ]:
adata.obs['Condition_Sample'] = adata.obs[['Group', 'Sample']].apply(lambda row: '_'.join(row.values.astype(str)), axis=1)


In [ ]:
benchmarker = meld.Benchmarker()

from joblib import Parallel, delayed

def simulate_pdf_calculate_likelihood(benchmarker, seed, beta):
    benchmarker.set_seed(seed)
    benchmarker.generate_ground_truth_pdf()
    
    benchmarker.generate_sample_labels()
    benchmarker.calculate_MELD_likelihood(beta=beta)
    MELD_mse = benchmarker.calculate_mse(benchmarker.expt_likelihood)
    return MELD_mse, seed, beta, benchmarker.graph.knn


In [ ]:
benchmarker.fit_phate(adata.obsm['X_pca_harmony']);

In [ ]:
knn_range = np.arange(1,25)
beta_range = np.arange(1,200)

In [ ]:
results = []

with Parallel(n_jobs=36) as p:
    for knn in knn_range:
        # doing this outside the parallel loop because building the graph takes the longest
        benchmarker.fit_graph(adata.X, knn=knn)
        print(knn)
        curr_results = p(delayed(simulate_pdf_calculate_likelihood)(benchmarker, seed, beta) \
                                       for seed in range(25) for beta in beta_range)
        curr_results = pd.DataFrame(curr_results, columns = ['MSE', 'seed', 'beta', 'knn'])
        results.append(curr_results)

results = pd.concat(results, axis=0)

In [ ]:
# giving dir_out path 
dir_out = "/dfs7/swaruplab/shared_lab/Collaborations/Wood/Reinstatement_Project/reinstatement_2022/MELD/"

In [ ]:
results.to_csv('{}meld_benchmarking_reinstatement.csv'.format(dir_out))

In [ ]:
print('test_finished')

In [ ]:
print('test_finished')

### Benchmarking

In [ ]:
results = pd.read_csv('{}meld_benchmarking_reinstatement.csv'.format(dir_out))

In [ ]:
results

In [ ]:
# We want to take the average of each set of random seeds for each combination of beta and knn values
results_wide = results.groupby(['beta', 'knn']).mean().sort_values(by='MSE').reset_index()

In [ ]:
results_wide

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
ax = scprep.plot.scatter(results_wide['beta'], results_wide['knn'], 
                         s=50, c=results_wide['MSE'], vmax=0.006, cmap='inferno_r')

# Highlight the top performing combination with a large red dot
top_result = results_wide.sort_values('MSE').iloc[0]
ax.scatter(top_result['beta'], top_result['knn'], c='r', s=100, linewidth=1, edgecolor='k')

# Save the figure as a PDF in the specified directory
fig_dir = '/dfs7/swaruplab/zechuas/Collaborations/Wood/reinstatement_2022/Analysis/MELD/figures/'  # Replace with your actual directory path
plt.savefig(f'{fig_dir}/scatter_plot.pdf', format='pdf')

# Optionally, show the plot
plt.show()

In [ ]:
adata.obs.Group

### Run MELD with best parameters

In [ ]:
meld_op = meld.MELD(beta=top_result['beta'])
sample_densities = meld_op.fit_transform(adata, sample_labels=adata.obs.Group)

In [ ]:
replicates = np.unique(adata.obs['Group'])
sample_likelihoods = sample_densities.copy()
for rep in replicates:
    curr_cols = sample_densities.columns[[col.startswith(rep) for col in sample_densities.columns]]
    scaler = sklearn.preprocessing.MinMaxScaler()
    sample_likelihoods[curr_cols] = scaler.fit_transform(sample_densities[curr_cols])


In [ ]:
sample_likelihoods

In [ ]:
adata.obs['Reinstatement_likelihood'] = sample_likelihoods['Reinstatement'].tolist()

In [ ]:
# set fig directory
fig_dir = "/dfs7/swaruplab/zechuas/Collaborations/Wood/reinstatement_2022/Analysis/MELD/figures/"
data_out = "/dfs7/swaruplab/shared_lab/Collaborations/Wood/Reinstatement_Project/reinstatement_2022/MELD/"
sc.settings.figdir = fig_dir

In [ ]:
sc.settings.set_figure_params(dpi=500, dpi_save=1000, figsize=(10,3), facecolor='white')
sc.pl.violin(adata,['Reinstatement_likelihood'], inner='box', size=0.5,  groupby='annotation', multi_panel=False, rotation=90, save = "_reinstatement_likelihood.pdf")


### Save the results

In [ ]:
adata.obs[['barcode', 'Reinstatement_likelihood']].to_csv('{}MELD_Reinstatement_likelihood_behavior.csv'.format(data_out))